In [42]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit
import os

path_root = "F:/collaborations/Tim_Hermans/aslc_intertidal"
os.chdir(path_root)

def load_datasets(path_measured, path_residuals, plot = False):

    df_measured = pd.read_csv(path_measured)
    w_measured = df_measured['surge']

    df_residuals = pd.read_csv(path_residuals)
    w_residuals = df_residuals['surge']
    t = pd.to_datetime(df_residuals['date'])
    t0 = t.iloc[0]
    t = (t - t0).dt.total_seconds() / 3600 # hours

    if plot:
        plt.plot(t, w_measured)
        plt.plot(t, w_residuals)
        plt.show()

        tmin, tmax = 15000, 20000
        plt.plot(t[tmin:tmax], w_residuals[tmin:tmax])
        plt.show()

    return t, w_measured, w_residuals
    
def estimate_stochastic_parameters(x, dt, xmean):
    '''dx = sigma * np.sqrt(dt) * N(0,1) + rho * (xmean - x) * dt'''

    def linear_fit(x, a, b):
        return a + x*b

    dx = np.diff(x)
    residuals = dx - xmean


    # Predict rho
    nonans = ~np.isnan(dx)
    par, _ = curve_fit(linear_fit, xdata = (xmean - x[:-1][nonans]), ydata = dx[nonans], p0 = (0, 1))
    rho_est = par[1] / dt  # (hr^-1)

    # Predict sigma
    sigma_est = np.std(residuals[nonans]) / np.sqrt(dt) # (m hr^-0.5)
    
    return rho_est, sigma_est

def generate_oun(nt, xmean = 0, sigma = 1, rho = 0.1, dt = 1):

    r = np.random.normal(size = nt)
    x = np.array([np.nan]*nt)
    x[0] = xmean
    for t in range(1, nt):
        x[t] = x[t - 1] + sigma*np.sqrt(dt)*r[t] + rho*(xmean - x[t - 1])*dt

    return x

def find_tidal_range(w_predicted, dt = 10/60):

    # Interpolate to 5 min measuring interval so 745 min tidal interval can be divided evenly
    t = np.arange(0, w_predicted.size, 1) * dt    
    tp = np.arange(0, t.max(), 5/60)
    w_predicted_interp = np.interp(x = tp, xp = t, fp = w_predicted)
    w_predicted_interp = w_predicted_interp[~np.isnan(w_predicted_interp)]

    # Reshape water level time series into stacked array of n tidal cycles (each 149 measurements long at dt = 5/60)
    n_measurements_per_tide = int((745/60) / (5/60))
    n_tides = int(np.floor(w_predicted_interp.size/n_measurements_per_tide))
    w_predicted_interp = w_predicted_interp[:int(n_tides*n_measurements_per_tide)]
    w_predicted_interp = w_predicted_interp.reshape(n_tides, n_measurements_per_tide)

    # Calcualte high & low water level of each tidal cycle, and from taht calculate the avg. tidal range
    high_water_level = w_predicted_interp.max(axis = 1)
    low_water_level = w_predicted_interp.min(axis = 1)
    tidal_range = (high_water_level - low_water_level)

    return tidal_range.mean()

files = ['papeete-015b-fra-uhslc', 'barcelona-bar-esp-cmems', 'sakai-ma53-jpn-jodc_jma']
for file in files:
    t, w_measured, w_residuals = load_datasets(path_measured = f'./data/waterlevel_timeseries/twl/{file}.csv', path_residuals = f'./data/waterlevel_timeseries/ntr/{file}.csv')

    dt = 60/60 # hours
    rho_est, sigma_est = estimate_stochastic_parameters(x = w_residuals, dt = dt, xmean = 0)

    avg_tidal_range = find_tidal_range(w_measured, dt = dt)

    # Rescale sigma by tidal amplitude
    sigma_est_rescaled = sigma_est / avg_tidal_range

    std_w = sigma_est / (np.sqrt(2 * rho_est))

    print(file)
    print(f'tidal_range = {avg_tidal_range:.3f}, sigma = {sigma_est:.3f}, sigma_rescaled = {sigma_est_rescaled:.3f}, rho = {rho_est:.3f}')
    print(f'SD w_residuals = {np.std(w_residuals):.3f}, SD w_residuals_est = {std_w:.3f}')
    print('')

    # w_residuals_est = generate_oun(nt = t.size, xmean = 0, sigma = sigma_est, rho = rho_est, dt = dt)
    # t = (t - t.min()) / (24*365.25) # years
    # plt.plot(t, w_residuals,     c = 'black')
    # plt.plot(t, w_residuals_est, c = 'red', alpha = 0.5)
    # plt.ylabel('Stochastic constituent of the water level (m)', fontsize = 12)
    # plt.ylabel('Stochastic constituent of the water level (m)', fontsize = 12)
    # plt.show()

    # plt.plot(t, w_measured, c = 'black')
    # plt.plot(t, w_measured - w_residuals + w_residuals_est, c = 'red', alpha = 0.5)
    # plt.ylabel('Measured water level (m NAP)', fontsize = 12)
    # plt.ylabel('Water level with predicted stochastic constituents (m NAP)', fontsize = 12)
    # plt.show()


papeete-015b-fra-uhslc
tidal_range = 0.195, sigma = 0.013, sigma_rescaled = 0.069, rho = 0.079
SD w_residuals = 0.034, SD w_residuals_est = 0.034

barcelona-bar-esp-cmems
tidal_range = 0.128, sigma = 0.007, sigma_rescaled = 0.056, rho = 0.005
SD w_residuals = 0.074, SD w_residuals_est = 0.075

sakai-ma53-jpn-jodc_jma
tidal_range = 0.174, sigma = 0.010, sigma_rescaled = 0.057, rho = 0.006
SD w_residuals = 0.090, SD w_residuals_est = 0.089

